In [1]:
from datetime import datetime, timezone
from src.helpers import get_project_folders, normalize_project
from src.metric import TimestampMetric
import orjson
import numpy as np

In [2]:
INPUT_FOLDER = "/home/jortvd/thesis-data/code-results-9-4-2026"
OUTPUT_FOLDER = "../results"

In [3]:
metric = TimestampMetric("time_inducing_to_fix_commit", True)

for project_folder in get_project_folders(INPUT_FOLDER):
    print(f"Processing {project_folder.name}...")
    if not (project_folder / "szz.json").exists():
        continue
    with open(project_folder / "szz.json", "rb") as f:
        szz = orjson.loads(f.read())

    for result in szz["results"]:
        to_time = result["date"]
        date = datetime.fromtimestamp(to_time, tz=timezone.utc)

        if any(file["new_path"].endswith(".rs") for file in result["modified_files"]):
            is_rust = True
        else:
            is_rust = False

        for commit in result["inducing_commits"]:
            if commit["trivial"]:
                continue
            from_time = commit["date"]
            metric.add(
                normalize_project(project_folder.name), 
                date, 
                (to_time - from_time) / (24 * 3600 * 30),
                is_rust
            )

metric.save(OUTPUT_FOLDER)

Processing BitBoxSwiss_bitbox02-firmware...
Processing KiwiTalk_KiwiTalk...
Processing NixOS_nixos-search...
Processing OISF_suricata...
Processing Pometry_Raphtory...
Processing SeaDve_Kooha...
Processing amir20_dtop...
Processing apache_arrow...
Processing apache_arrow-rs...
Processing appsinacup_godot-rapier-physics...
Processing ariebovenberg_whenever...
Processing brndnmtthws_dryoc...
Processing cjdelisle_cjdns...
Processing cooklang_cookcli...
Processing coreboot_coreboot...
Processing emmericp_ixy...
Processing eraserhd_parinfer-rust...
Processing fosskers_aura...
Processing gschup_ggrs...
Processing intentee_paddler...
Processing ixy-languages_ixy.rs...
Processing jedisct1_libsodium...
Processing jely2002_youtube-dl-gui...
Processing jneem_nnnoiseless...
Processing kkoomen_vim-doge...
Processing kubetail-org_kubetail...
Processing libjxl_jxl-rs...
Processing libjxl_libjxl...
Processing m-labs_artiq...
Processing madler_zlib...
Processing max-baz_wluma...
Processing memorysafety

In [ ]:
metric = RatioMetric("fix_inducing_commit_ratio", True)

for project_folder in get_project_folders(INPUT_FOLDER):
    if not (project_folder / "szz.json").exists():
        continue
    with open(project_folder / "szz.json") as f:
        szz = orjson.loads(f.read())
    
    inducing_commits = set()

    for result in szz["results"]:
        for commit in result["inducing_commits"]:
            if commit["trivial"]:
                continue
            hash = commit["hash"]
            if hash in inducing_commits:
                continue
            inducing_commits.add(hash)
    
    with open(project_folder / "commit_stats.json") as f:
        commits = orjson.loads(f.read())

    for commit in commits:
        date = datetime.fromtimestamp(commit["author_date"], tz=timezone.utc)
        has_rust = any(file["new_path"].endswith(".rs") for file in commit["file_details"])
        
        if commit["hash"] in inducing_commits:
            metric.add_num(
                normalize_project(project_folder.name),
                date,
                1,
                has_rust
            )
        metric.add_den(
            normalize_project(project_folder.name),
            date,
            1,
            has_rust
        )

metric.save(OUTPUT_FOLDER)

In [ ]:
metric = RatioMetric("fix_commit_ratio", True)

for project_folder in get_project_folders(INPUT_FOLDER):
    if not (project_folder / "szz.json").exists():
        continue
    with open(project_folder / "szz.json") as f:
        szz = orjson.loads(f.read())
    
    fix_commits = set()

    for result in szz["results"]:
        hash = result["hash"]
        if hash in fix_commits:
            continue
        fix_commits.add(hash)
    
    with open(project_folder / "commit_stats.json") as f:
        commits = orjson.loads(f.read())

    for commit in commits:
        date = datetime.fromtimestamp(commit["author_date"], tz=timezone.utc)
        has_rust = any(file["new_path"].endswith(".rs") for file in commit["file_details"])
        
        if commit["hash"] in fix_commits:
            metric.add_num(
                normalize_project(project_folder.name),
                date,
                1,
                has_rust
            )
        metric.add_den(
            normalize_project(project_folder.name),
            date,
            1,
            has_rust
        )

metric.save(OUTPUT_FOLDER)

In [ ]:
metric = RatioMetric("fix_and_inducing_commit_ratio", True)

for project_folder in get_project_folders(INPUT_FOLDER):
    if not (project_folder / "szz.json").exists():
        continue
    with open(project_folder / "szz.json") as f:
        szz = orjson.loads(f.read())
    
    fix_commits = set()
    inducing_commits = set()

    for result in szz["results"]:
        fix_commits.add(result["hash"])

        for commit in result["inducing_commits"]:
            if commit["trivial"]:
                continue
            inducing_commits.add(commit["hash"])

    fix_and_inducing_commits = fix_commits.intersection(inducing_commits)
    
    with open(project_folder / "commit_stats.json") as f:
        commits = orjson.loads(f.read())

    for commit in commits:
        date = datetime.fromtimestamp(commit["author_date"], tz=timezone.utc)
        has_rust = any(file["new_path"].endswith(".rs") for file in commit["file_details"])

        if commit["hash"] in fix_and_inducing_commits:
            metric.add_num(
                normalize_project(project_folder.name),
                date,
                1,
                has_rust
            )

        metric.add_den(
            normalize_project(project_folder.name),
            date,
            1,
            has_rust
        )

metric.save(OUTPUT_FOLDER)